# 23 Distributions: every `dist.` word against R and the textbook formulas, and fits to real data

`sdata.distribution` with its parameters and every word of the distribution it gives (`find_density_at`,
`find_chance_that_value`, `find_percentile`, `find_cut_off`, `find_middle_range`, `find_average`, `find_median`,
`find_variance`, `find_standard_deviation`, `show_summary`, `show_parameters`, `make_random_numbers`,
`turn_into_table`), on 15 distributions; fits with `table.fit_distribution_to` and `nums.fit_distribution`
(`fixed=`), and `table.compare_distributions_for`.

**Only simplydata runs in this notebook.** The reference values (`references/23_distributions.csv`, from
`scripts/references/23_distributions.py`): R's density and quantile functions (`dnorm`, `qgamma`, `qbinom` ...,
R circular `dvonmises`), the moments from their textbook formulas (Johnson, Kotz & Balakrishnan 1994, 1995;
Johnson, Kemp & Kotz 2005), SciPy's draws with the same seed and the true mean within 4 Monte Carlo standard
errors, the likelihood equations solved in SciPy, closed-form estimates and their standard errors, R MASS
`fitdistr` and fitdistrplus `fitdist` run to a relative tolerance of 1e-14, and the goodness-of-fit statistics
written out. `dist.draw_chart`: the curve it draws, read back with `chart.get_numbers_drawn()`, against R's
density and distribution functions (`dnorm`, `pnorm` ...) at the same x (relative difference 1e-10).

Data: the daily ozone of airquality and the yearly counts of great inventions (`discoveries`), both from R.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_numbers, read_row, read_rows, versions

NOTEBOOK = "23_distributions"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The 15 distributions

Each with simplydata's name and parameters (R's names, which simplydata also accepts, where they are shorter).

In [2]:
CASES = [("normal", dict(average=50, standard_deviation=10), [35, 50, 61.5], False),
         ("lognormal", dict(meanlog=1.0, sdlog=0.5), [0.5, 2.7, 6], False),
         ("exponential", dict(rate=0.5), [0.1, 2, 7], False),
         ("gamma", dict(shape=2, rate=0.5), [0.3, 4, 11], False),
         ("beta", dict(shape1=2.0, shape2=5.0), [0.05, 0.3, 0.8], False),
         ("uniform", dict(min=2, max=5), [2.5, 3.5, 4.9], False),
         ("t", dict(degrees_of_freedom=7), [-2.5, 0, 1.3], False),
         ("chi-square", dict(degrees_of_freedom=4), [0.5, 3, 12], False),
         ("F", dict(df1=5, df2=12), [0.2, 1, 4], False),
         ("weibull", dict(shape=1.5, scale=2.0), [0.2, 1.7, 5], False),
         ("logistic", dict(location=1, scale=2), [-4, 1, 6], False),
         ("binomial", dict(tries=10, chance=0.3), [0, 3, 7], True),
         ("poisson", dict(average=3), [0, 3, 8], True),
         ("negative binomial", dict(size=3, prob=0.4), [0, 4, 12], True),
         ("geometric", dict(chance=0.25), [0, 2, 9], True)]
PLAIN = {"meanlog": "log_average", "sdlog": "log_standard_deviation", "shape1": "shape_1", "shape2": "shape_2",
         "df1": "degrees_of_freedom_1", "df2": "degrees_of_freedom_2", "size": "size", "prob": "chance",
         "min": "lowest", "max": "highest"}

for name, params, values, counts in CASES:
    dist = sdata.distribution(name, **params)
    ds = f"{name} " + ", ".join(f"{k}={v:g}" for k, v in params.items())
    summary = read_row(quiet(dist.show_summary))
    checks.check(ds, "dist.find_average", "average", quiet(dist.find_average))
    checks.check(ds, "dist.find_variance", "variance", quiet(dist.find_variance))
    checks.check(ds, "dist.find_standard_deviation", "standard deviation", quiet(dist.find_standard_deviation))
    checks.check_values(ds, "dist.show_summary", {f"show_summary {k}": v for k, v in summary.items()})
    for x in values:
        if counts:
            checks.check(ds, "dist.find_chance_that_value (a count)", f"at {x:g}", quiet(dist.find_chance_that_value, x))
        else:
            checks.check(ds, "dist.find_density_at", f"at {x:g}", quiet(dist.find_density_at, x))
    for p, ours in zip([5, 50, 95], read_numbers(quiet(dist.find_percentile, [5, 50, 95]))):
        checks.check(ds, "dist.find_percentile (percent=)", f"percentile {p}", ours)
    checks.check(ds, "dist.find_median", "median", quiet(dist.find_median))
    checks.check(ds, "dist.find_cut_off (with_chance_below=)", "cut-off with 20 % below", quiet(dist.find_cut_off, with_chance_below=0.2))
    if not counts:
        checks.check(ds, "dist.find_cut_off (with_chance_above=)", "cut-off with 20 % above", quiet(dist.find_cut_off, with_chance_above=0.2))
    for percent, method in [(90, "dist.find_middle_range (percent=90)"), (95, "dist.find_middle_range")]:
        low, high = read_numbers(quiet(dist.find_middle_range, percent))
        checks.check_values(ds, method, {f"middle {percent} % low": low, f"middle {percent} % high": high})
    shown = dist.show_parameters()
    for key, value in params.items():
        checks.check(ds, "dist.show_parameters", f"{key} is shown (as {PLAIN.get(key, key)} or its plain name)",
                     float(any(v == value for v in shown.values())))
    checks.check(ds, "dist.make_random_numbers (how_many=, with_seed=)", "1000 draws",
                 read_numbers(quiet(dist.make_random_numbers, how_many=1000, with_seed=3)))
    checks.check(ds, "dist.make_random_numbers", "average of 400 000 draws",
                 quiet(quiet(dist.make_random_numbers, how_many=400_000, with_seed=4).find_average))

vm = sdata.distribution("von mises", average=1, concentration=2)
for x in (0.0, 1.0, 2.5, 5.0):
    checks.check("von mises average=1, concentration=2", "dist.find_density_at", f"at {x:g} radians", quiet(vm.find_density_at, x))
checks.check("von mises average=1, concentration=2", "dist.find_density_at", "at 1 radian", quiet(vm.find_density_at, 1.0))

agrees  normal average=50, standard_deviation=10 | dist.find_average | average: simplydata 50, textbook formula (Johnson, Kotz & Balakrishnan) 50; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook formula (Johnson, Kotz & Balakrishnan) (run reference)
agrees  normal average=50, standard_deviation=10 | dist.find_variance | variance: simplydata 100, textbook formula (Johnson, Kotz & Balakrishnan) 100; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook formula (Johnson, Kotz & Balakrishnan) (run reference)
agrees  normal average=50, standard_deviation=10 | dist.find_standard_deviation | standard deviation: simplydata 10, textbook formula (Johnson, Kotz & Balakrishnan) 10; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook formula (Johnson, Kotz & Balakrishnan) (run reference)
agrees  normal average=50, standard_deviation=10 | dist.show_summary | show_summary average: simplydata 50, textbook formula (Johnson, Kotz & Balakrishnan) 50; rela

agrees  t degrees_of_freedom=7 | dist.find_cut_off (with_chance_below=) | cut-off with 20 % below: simplydata -0.896029644314, R qt -0.896029644314; relative difference 0.00e+00 (allowed 1e-10)
      Source: R qt (run reference)
agrees  t degrees_of_freedom=7 | dist.find_cut_off (with_chance_above=) | cut-off with 20 % above: simplydata 0.896029644314, R qt 0.896029644314; relative difference 2.48e-16 (allowed 1e-10)
      Source: R qt (run reference)
agrees  t degrees_of_freedom=7 | dist.find_middle_range (percent=90) | middle 90 % low: simplydata -1.89457860509, R qt -1.89457860509; relative difference 1.17e-16 (allowed 1e-10)
      Source: R qt (run reference)
agrees  t degrees_of_freedom=7 | dist.find_middle_range (percent=90) | middle 90 % high: simplydata 1.89457860509, R qt 1.89457860509; relative difference 1.17e-16 (allowed 1e-10)
      Source: R qt (run reference)
agrees  t degrees_of_freedom=7 | dist.find_middle_range | middle 95 % low: simplydata -2.36462425159, R qt -2.364

agrees  F df1=5, df2=12 | dist.find_cut_off (with_chance_above=) | cut-off with 20 % above: simplydata 1.74034099166, R qf 1.74034099166; relative difference 1.28e-16 (allowed 1e-10)
      Source: R qf (run reference)
agrees  F df1=5, df2=12 | dist.find_middle_range (percent=90) | middle 90 % low: simplydata 0.21378010334, R qf 0.21378010334; relative difference 9.09e-16 (allowed 1e-10)
      Source: R qf (run reference)
agrees  F df1=5, df2=12 | dist.find_middle_range (percent=90) | middle 90 % high: simplydata 3.10587523908, R qf 3.10587523908; relative difference 1.43e-16 (allowed 1e-10)
      Source: R qf (run reference)
agrees  F df1=5, df2=12 | dist.find_middle_range | middle 95 % low: simplydata 0.153267293494, R qf 0.153267293494; relative difference 7.24e-16 (allowed 1e-10)
      Source: R qf (run reference)
agrees  F df1=5, df2=12 | dist.find_middle_range | middle 95 % high: simplydata 3.8911339339, R qf 3.8911339339; relative difference 2.28e-16 (allowed 1e-10)
      Source:

agrees  poisson average=3 | dist.make_random_numbers | average of 400 000 draws: simplydata 3.0029625, true mean (4 MCSE) 3; difference 2.96e-03 (allowed 1e-02)
      Source: true mean (4 MCSE) (run reference)
agrees  negative binomial size=3, prob=0.4 | dist.find_average | average: simplydata 4.5, textbook formula (Johnson, Kotz & Balakrishnan) 4.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook formula (Johnson, Kotz & Balakrishnan) (run reference)
agrees  negative binomial size=3, prob=0.4 | dist.find_variance | variance: simplydata 11.25, textbook formula (Johnson, Kotz & Balakrishnan) 11.25; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook formula (Johnson, Kotz & Balakrishnan) (run reference)
agrees  negative binomial size=3, prob=0.4 | dist.find_standard_deviation | standard deviation: simplydata 3.35410196625, textbook formula (Johnson, Kotz & Balakrishnan) 3.35410196625; relative difference 0.00e+00 (allowed 1e-12)
      Source: textbook

agrees  geometric chance=0.25 | dist.make_random_numbers | average of 400 000 draws: simplydata 3.00337, true mean (4 MCSE) 3; difference 3.37e-03 (allowed 2e-02)
      Source: true mean (4 MCSE) (run reference)
agrees  von mises average=1, concentration=2 | dist.find_density_at | at 0 radians: simplydata 0.205714499516, R circular dvonmises 0.205714499516; relative difference 2.70e-16 (allowed 1e-12)
      Source: R circular dvonmises (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/dvonmises.html (version 0.5.2)
agrees  von mises average=1, concentration=2 | dist.find_density_at | at 1 radians: simplydata 0.515885412019, R circular dvonmises 0.515885412019; relative difference 2.15e-16 (allowed 1e-12)
      Source: R circular dvonmises (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/dvonmises.html (version 0.5.2)
agrees  von mises average=1, concentration=2 | dist.find_density_at | at 2.5 radians: simplydata 0.0804277346011, R circul

1

## Fits to real data: daily ozone (airquality) and great inventions (discoveries)

In [3]:
def named(row):
    """The numbers of a turn_into_table row, under the names the references use (log_average ...)."""
    values = {}
    for key, value in row.items():
        for plain in ("log standard deviation", "log average", "standard deviation"):
            key = key.replace(plain, plain.replace(" ", "_"))
        values[key] = value
    return values


air = quiet(sdata.table, data_file(NOTEBOOK, "airquality"), separator=",")
ozone = air.get_column("Ozone")
ds = "airquality ozone (Chambers et al. 1983)"
for name in ("normal", "lognormal", "exponential", "gamma", "weibull"):
    fit = quiet(air.fit_distribution_to, "Ozone", name)
    checks.check_values(ds, f'table.fit_distribution_to ("{name}")', named(read_row(quiet(fit.turn_into_table))))
    checks.check_values(ds, f'nums.fit_distribution (distribution="{name}")',
                        named(quiet(ozone.fit_distribution, distribution=name).show_parameters()))
for word, fit in [("table.fit_distribution_to", quiet(air.fit_distribution_to, "Ozone", "gamma", fixed={"shape": 2})),
                  ("nums.fit_distribution", quiet(ozone.fit_distribution, "gamma", fixed={"shape": 2}))]:
    checks.check_values(ds, f'{word} ("gamma", fixed={{"shape": 2}})', read_row(quiet(fit.turn_into_table)))

inventions = quiet(sdata.table, data_file(NOTEBOOK, "discoveries_counts"), separator=",").get_column("count")
ds = "discoveries 1860-1959 (R datasets)"
row = read_row(quiet(quiet(inventions.fit_distribution, distribution="poisson").turn_into_table))
parameter = [c for c in row if c in ("average", "lambda", "rate")][0]
checks.check_values(ds, 'nums.fit_distribution (distribution="poisson")',
                    {"average": row[parameter], "standard error": row[f"standard error of {parameter}"]})
checks.check_values(ds, 'nums.fit_distribution (distribution="negative binomial")',
                    read_row(quiet(quiet(inventions.fit_distribution, distribution="negative binomial").turn_into_table)))

agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("normal") | average: simplydata 42.1293103448, closed-form MLE 42.1293103448; relative difference 1.69e-16 (allowed 1e-12)
      Source: closed-form MLE (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("normal") | average: simplydata 42.1293103448, R fitdistrplus fitdist (reltol 1e-14) 42.1293103448; relative difference 1.69e-16 (allowed 1e-06)
      Source: R fitdistrplus fitdist (reltol 1e-14) (run reference) - https://search.r-project.org/CRAN/refmans/fitdistrplus/html/fitdist.html (version 1.2.6)
agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("normal") | standard_deviation: simplydata 32.8453875869, closed-form MLE 32.8453875869; relative difference 0.00e+00 (allowed 1e-12)
      Source: closed-form MLE (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("normal") | standard_deviation: simplyd

agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("gamma", fixed={"shape": 2}) | rate: simplydata 0.0474728873487, 2 / mean (closed form) 0.0474728872519; relative difference 2.04e-09 (allowed 1e-06)


      Source: 2 / mean (closed form) (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("gamma", fixed={"shape": 2}) | standard error of rate: simplydata 0.00311674767079, rate / sqrt(2 n) 0.0031167477413; relative difference 2.26e-08 (allowed 1e-06)
      Source: rate / sqrt(2 n) (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.fit_distribution_to ("gamma", fixed={"shape": 2}) | log-likelihood: simplydata -542.494642246, at the closed-form rate -542.494642246; relative difference 2.10e-16 (allowed 1e-06)
      Source: at the closed-form rate (run reference)
agrees  airquality ozone (Chambers et al. 1983) | nums.fit_distribution ("gamma", fixed={"shape": 2}) | rate: simplydata 0.0474728873487, 2 / mean (closed form) 0.0474728872519; relative difference 2.04e-09 (allowed 1e-06)
      Source: 2 / mean (closed form) (run reference)
agrees  airquality ozone (Chambers et al. 1983) | nums.fit_distribution ("gamma", fixed={"sha

2

## `compare_distributions_for` on the daily ozone

In [4]:
values = {}
for row in read_rows(quiet(air.compare_distributions_for, "Ozone")):
    for column in row:
        values[f"{row['distribution']}: {column}"] = row[column]
checks.check_values("airquality ozone (Chambers et al. 1983)", "table.compare_distributions_for", values)

agrees  airquality ozone (Chambers et al. 1983) | table.compare_distributions_for | gamma: log-likelihood: simplydata -541.537642446, formula at the reference fit, NumPy -541.537642446; relative difference 0.00e+00 (allowed 1e-06)
      Source: formula at the reference fit, NumPy (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.compare_distributions_for | gamma: AIC: simplydata 1087.07528489, formula at the reference fit, NumPy 1087.07528489; relative difference 0.00e+00 (allowed 1e-06)
      Source: formula at the reference fit, NumPy (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.compare_distributions_for | gamma: BIC: simplydata 1092.58246527, formula at the reference fit, NumPy 1092.58246527; relative difference 0.00e+00 (allowed 1e-06)
      Source: formula at the reference fit, NumPy (run reference)
agrees  airquality ozone (Chambers et al. 1983) | table.compare_distributions_for | gamma: Anderson-Darling statistic: simplydata 0.737

36

## The drawn curves: `dist.draw_chart`

The density (and with `show="cumulative"` the distribution function) that `dist.draw_chart` draws for each of the 11
continuous distributions, read back with `chart.get_numbers_drawn()` (the y of the drawn line) and compared, point by
point, with R's `d...` and `p...` functions at the same x, by the largest relative difference.

In [5]:
for name, params, values, counts in CASES:
    if counts:
        continue
    dist = sdata.distribution(name, **params)
    ds = f"{name} " + ", ".join(f"{k}={v:g}" for k, v in params.items())
    for show, method in (("density", "dist.draw_chart"), ("cumulative", 'dist.draw_chart (show="cumulative")')):
        chart = quiet(dist.draw_chart) if show == "density" else quiet(dist.draw_chart, show="cumulative")
        drawn = quiet(chart.get_numbers_drawn)
        checks.check(ds, method, "the drawn curve", read_numbers(drawn.get_column("y")))

agrees  normal average=50, standard_deviation=10 | dist.draw_chart | the drawn curve: largest relative difference over 200 values: simplydata 2.46742479517e-16, R dnorm 0; difference 2.47e-16 (allowed 1e-10)
      Source: R dnorm (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html (R version 4.6.2 (2026-10-29))
agrees  normal average=50, standard_deviation=10 | dist.draw_chart (show="cumulative") | the drawn curve: largest relative difference over 200 values: simplydata 1.59092004259e-15, R pnorm 0; difference 1.59e-15 (allowed 1e-10)
      Source: R pnorm (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html (R version 4.6.2 (2026-10-29))
agrees  lognormal meanlog=1, sdlog=0.5 | dist.draw_chart | the drawn curve: largest relative difference over 200 values: simplydata 2.3310893401e-15, R dlnorm 0; difference 2.33e-15 (allowed 1e-10)
      Source: R dlnorm (run reference) - https://search.r-project.org/R/refmans/stats/html/Lognormal

agrees  gamma shape=2, rate=0.5 | dist.draw_chart (show="cumulative") | the drawn curve: largest relative difference over 200 values: simplydata 1.09920210381e-15, R pgamma 0; difference 1.10e-15 (allowed 1e-10)
      Source: R pgamma (run reference) - https://search.r-project.org/R/refmans/stats/html/GammaDist.html (R version 4.6.2 (2026-10-29))
agrees  beta shape1=2, shape2=5 | dist.draw_chart | the drawn curve: largest relative difference over 200 values: simplydata 1.9397920262e-15, R dbeta 0; difference 1.94e-15 (allowed 1e-10)
      Source: R dbeta (run reference) - https://search.r-project.org/R/refmans/stats/html/Beta.html (R version 4.6.2 (2026-10-29))
agrees  beta shape1=2, shape2=5 | dist.draw_chart (show="cumulative") | the drawn curve: largest relative difference over 200 values: simplydata 1.3576132666e-15, R pbeta 0; difference 1.36e-15 (allowed 1e-10)
      Source: R pbeta (run reference) - https://search.r-project.org/R/refmans/stats/html/Beta.html (R version 4.6.2 (20

## Result

In [6]:
checks.finish()

23_distributions: 478 comparisons - 475 agrees, 3 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,23_distributions,"normal average=50, standard_deviation=10",dist.find_average,average,5.000000e+01,50.0,"textbook formula (Johnson, Kotz & Balakrishnan)",0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,23_distributions,"normal average=50, standard_deviation=10",dist.find_variance,variance,1.000000e+02,100.0,"textbook formula (Johnson, Kotz & Balakrishnan)",0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,23_distributions,"normal average=50, standard_deviation=10",dist.find_standard_deviation,standard deviation,1.000000e+01,10.0,"textbook formula (Johnson, Kotz & Balakrishnan)",0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,23_distributions,"normal average=50, standard_deviation=10",dist.show_summary,show_summary average,5.000000e+01,50.0,"textbook formula (Johnson, Kotz & Balakrishnan)",0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,23_distributions,"normal average=50, standard_deviation=10",dist.show_summary,show_summary variance,1.000000e+02,100.0,"textbook formula (Johnson, Kotz & Balakrishnan)",0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
473,23_distributions,"F df1=5, df2=12","dist.draw_chart (show=""cumulative"")",the drawn curve: largest relative difference o...,1.118017e-15,0.0,R pf,1.118017e-15,1.000000e-10,absolute,1.118017e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
474,23_distributions,"weibull shape=1.5, scale=2",dist.draw_chart,the drawn curve: largest relative difference o...,9.629355e-16,0.0,R dweibull,9.629355e-16,1.000000e-10,absolute,9.629355e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
475,23_distributions,"weibull shape=1.5, scale=2","dist.draw_chart (show=""cumulative"")",the drawn curve: largest relative difference o...,2.831831e-16,0.0,R pweibull,2.831831e-16,1.000000e-10,absolute,2.831831e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
476,23_distributions,"logistic location=1, scale=2",dist.draw_chart,the drawn curve: largest relative difference o...,6.682631e-16,0.0,R dlogis,6.682631e-16,1.000000e-10,absolute,6.682631e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
